# Phase 2: Data Modeling (Merging)

In [2]:
# loading for data 

import pandas as pd

customers = pd.read_csv("../data/processed/customers_clean.csv")
products  = pd.read_csv("../data/processed/products_clean.csv")
sales     = pd.read_csv("../data/processed/sales_transactions_clean.csv")

print(customers.shape)
print(products.shape)
print(sales.shape)


(100, 4)
(20, 3)
(1000, 7)


# Check Keys Before Merge

In [3]:
customers["CustomerID"].nunique(), customers.shape[0]


(100, 100)

In [4]:
products["ProductID"].nunique(), products.shape[0]


(20, 20)

In [5]:
sales["TransactionID"].nunique(), sales.shape[0]

(1000, 1000)

In [6]:
missing_customers = set(sales["CustomerID"]) - set(customers["CustomerID"])
missing_products  = set(sales["ProductID"]) - set(products["ProductID"])

len(missing_customers), len(missing_products)

(1, 0)

In [7]:
list(missing_customers)[:10] # ['CUST-999']

list(missing_products)[:10] # nothing 


[]

# marge all tables

In [8]:
df = sales.merge(customers, on="CustomerID", how="left")
df = df.merge(products, on="ProductID", how="left")

df.shape #(1000,12) all rows transactions are present and all columns


(1000, 12)

In [9]:
df.head()


,TransactionID,CustomerID,ProductID,Date,Quantity,Discount,Total_Amount,Name,Region,Signup_Date,Category,Price
0,10001,CUST-055,PROD-008,2023-01-01 00:00:00,9,0.2,1355.76,Customer_55,West,2021-01-17,Electronics,188.30
1,10002,CUST-023,PROD-019,2023-01-01 01:00:00,1,0.1,NaN,Customer_23,South,2020-06-07,Electronics,193.67
2,10003,CUST-095,PROD-002,2023-01-01 02:00:00,9,0.0,3552.48,Customer_95,Midwest,2021-10-24,Clothing,394.72
3,10004,CUST-058,PROD-019,2023-01-01 03:00:00,2,0.2,309.87,Customer_58,East,2021-02-07,Electronics,193.67
4,10005,CUST-070,PROD-003,2023-01-01 04:00:00,8,0.1,1147.39,Customer_70,East,2021-05-02,Office,159.36


In [10]:
df.isna().sum()


TransactionID     0
CustomerID        0
ProductID         0
Date              0
Quantity          0
Discount          0
Total_Amount     82
Name             20
Region           20
Signup_Date      20
Category          0
Price             0
dtype: int64

# Handling Data are NaN 


In [11]:
# As ['CUST-999'] has dirty data after marge 
#  ---- handle missing customers by filling with 'Unknown' 
# Name             20
# Region           20
# Signup_Date      20

df['Name'] = df['Name'].fillna('Unknown')
df['Region'] =df['Region'].fillna('Unknown')
df['Signup_Date'] =df['Signup_Date'].fillna(pd.NaT)


df.isna().sum()
df[df["Signup_Date"].isna()]

,TransactionID,CustomerID,ProductID,Date,Quantity,Discount,Total_Amount,Name,Region,Signup_Date,Category,Price
234,10235,CUST-999,PROD-016,2023-01-10 18:00:00,8,0.20,2737.98,Unknown,Unknown,NaN,Furniture,427.81
235,10236,CUST-999,PROD-020,2023-01-10 19:00:00,7,0.05,913.58,Unknown,Unknown,NaN,Electronics,137.38
273,10274,CUST-999,PROD-019,2023-01-12 09:00:00,2,0.05,367.97,Unknown,Unknown,NaN,Electronics,193.67
276,10277,CUST-999,PROD-013,2023-01-12 11:00:00,2,0.00,513.88,Unknown,Unknown,NaN,Clothing,256.94
384,10385,CUST-999,PROD-008,2023-01-17 00:00:00,8,0.20,1205.12,Unknown,Unknown,NaN,Electronics,188.30
397,10398,CUST-999,PROD-009,2023-01-17 13:00:00,3,0.10,899.56,Unknown,Unknown,NaN,Furniture,333.17
413,10414,CUST-999,PROD-004,2023-01-18 05:00:00,4,0.00,1342.12,Unknown,Unknown,NaN,Office,335.53
557,10558,CUST-999,PROD-008,2023-01-24 05:00:00,6,0.05,1073.31,Unknown,Unknown,NaN,Electronics,188.30
561,10562,CUST-999,PROD-001,2023-01-24 09:00:00,8,0.10,1211.54,Unknown,Unknown,NaN,Clothing,168.27
599,10600,CUST-999,PROD-006,2023-01-25 23:00:00,8,0.10,825.55,Unknown,Unknown,NaN,Electronics,114.66


In [12]:
#Handle total amount by calculating it from Quantity and Price Discounting any missing values in Quantity or Price as zero for the calculation.
df["Calculated_Revenue"] = (
    df["Quantity"] * df["Price"] * (1 - df["Discount"])
)

df[["Total_Amount","Calculated_Revenue"]].isna().sum()


Total_Amount          82
Calculated_Revenue     0
dtype: int64

In [13]:
df["Calculated_Revenue"].describe()

count      1000.000000
mean       2374.907299
std       21068.867244
min       -2959.803000
25%         379.380000
50%         825.552000
75%        1435.396000
max      422829.000000
Name: Calculated_Revenue, dtype: float64

In [14]:
df["Total_Amount"].describe()

count       918.000000
mean       2498.725969
std       21985.083289
min       -2959.800000
25%         387.160000
50%         825.550000
75%        1437.705000
max      422829.000000
Name: Total_Amount, dtype: float64

# Compare with Dirty Total_Amount

In [15]:
df["Difference"] = df["Total_Amount"] - df["Calculated_Revenue"]
df["Difference"].describe()

df["Difference"].isna().sum()

np.int64(82)

# Time se

In [17]:
# Convert 'Date' to datetime explicitly
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Extract Month and Year for aggregation and plotting
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Year"] = df["Date"].dt.year
# Create a new column with month names
df["Month_Name"] = df["Date"].dt.month_name()


df["Year"] = df["Date"].dt.year

# Preview
df[["Date", "Month_Name", "Year"]].head(10)


,Date,Month_Name,Year
0,2023-01-01 00:00:00,January,2023
1,2023-01-01 01:00:00,January,2023
2,2023-01-01 02:00:00,January,2023
3,2023-01-01 03:00:00,January,2023
4,2023-01-01 04:00:00,January,2023
5,2023-01-01 05:00:00,January,2023
6,2023-01-01 06:00:00,January,2023
7,2023-01-01 07:00:00,January,2023
8,2023-01-01 08:00:00,January,2023
9,2023-01-01 09:00:00,January,2023
